## HOT Station ALOHA -- Bottle Data

In this example, we use the Hawaii Ocean Time-series (HOT) bottle data to explore a classic problem in observational oceanography: many scientifically important variables are measured infrequently, while other measurements are nearly complete.
Can we use well-sampled variables to reconstruct sparse ones?

We will work with data from CMAP (`tblHOT_Bottle_ALOHA`), which contains bottle measurements from Station ALOHA spanning 1988 to present at depths from the surface to 30 m.

Learning outcomes for this tutorial:

- You should be able to perform exploratory data analysis on a real oceanographic dataset, including assessing missingness and identifying predictor-target relationships
- You should be able to fit a linear regression baseline and interpret its performance
- You should be able to build, train, and evaluate a feedforward neural network for a regression task
- You should be able to compare NN performance against the linear baseline and reason about when added model complexity is justified
- You should be able to use a trained network to fill observational gaps and critically assess the result

### Import Packages

In [ ]:
import pycmap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import tensorflow as tf

# ── Helpers ──────────────────────────────────────────────────────────\
# compute R²
def r2_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1 - ss_res / ss_tot

def denorm(y_norm, y_mean, y_std):
    return y_norm * y_std + y_mean

### CMAP Query

In [ ]:
api = pycmap.API(token='<YOUR TOKEN HERE>')

table_name = 'tblHOT_Bottle_ALOHA'
df = api.query(f"""
    SELECT * FROM {table_name}
    WHERE depth BETWEEN 0 AND 30
    ORDER BY time
""")

# Alternatively, just load the data from the CSV
# df.to_csv('hot_aloha_25m_full.csv', index=False)
# df = pd.read_csv('../data/hot_aloha/hot_aloha_25m_full.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head())

## EDA

Begin by performing exploratory data analysis so that you have an expectation of how the data are structured and can make informed choices about which variables to model.


### Check Depth and Time Range
Confirm that our query returned the expected spatiotemporal coverage.

In [ ]:
print(f'Depth range: {df["depth"].min():.1f} to {df["depth"].max():.1f} m')
print(f'Depth mean:  {df["depth"].mean():.1f} m')
print(f'Depth std:   {df["depth"].std():.1f} m')

df['time'] = pd.to_datetime(df['time'])
print(f'\nTime range: {df["time"].min()} to {df["time"].max()}')
print(f'Number of unique dates: {df["time"].dt.date.nunique()}')
print(f'\nLat: {df["lat"].unique()}')
print(f'Lon: {df["lon"].unique()}')

### Missingness

The HOT program has sampled Station ALOHA roughly monthly since October 1988, but not every variable is measured on every cast or every cruise.
Before selecting variables for modeling, we need to understand which variables have enough data to work with.

In [ ]:
# ── Missingness by variable ──────────────────────────────────────────
meta_cols = ['time', 'lat', 'lon', 'depth', 'cruise_name', 'date_time', 'botid', 'press']
var_cols  = [c for c in df.columns if c not in meta_cols]

miss = df[var_cols].isna().mean().sort_values(ascending=False) * 100
print(f"Total measured variables: {len(var_cols)}")
print(f"Variables with >90% missing: {(miss > 90).sum()}")
print(f"Variables with <50% missing: {(miss < 50).sum()}\n")

# Data availability for each variable
avail_pct = (100 - miss).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 12))
ax.barh(avail_pct.index, avail_pct.values)
ax.axvline(10, color='orange', linestyle='--', alpha=0.7, label='10% present')
ax.axvline(50, color='red',    linestyle='--', alpha=0.7, label='50% present')
ax.set_xlabel('Data Available (%)')
ax.set_title('Data Availability by Variable (0-30 m)')
ax.legend()
plt.tight_layout()
plt.show()

### Correlation Structure

We compute pairwise correlations for variables with sufficient observations. Look for variable pairs with strong positive or negative correlations.
These suggest that one variable could be predicted from another, especially when the predictor is well-sampled and the target is sparse.

In [ ]:
# ── Correlation matrix of well-sampled variables ─────────────────────
min_obs = # modify: try different thresholds and see how the correlation matrix changes
candidates = [c for c in var_cols if df[c].notna().sum() >= min_obs]
print(f"Variables with >= {min_obs} observations: {len(candidates)}")
print(candidates)

corr = df[candidates].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(candidates)))
ax.set_xticklabels(candidates, rotation=90, fontsize=8)
ax.set_yticks(range(len(candidates)))
ax.set_yticklabels(candidates, fontsize=8)
for i in range(len(candidates)):
    for j in range(len(candidates)):
        val = corr.iloc[i, j]
        if abs(val) > 0.3:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6,
                    color='white' if abs(val) > 0.7 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title(f'Pairwise correlations (0-30 m, variables with >= {min_obs} obs)')
plt.tight_layout()
plt.show()

### Exercise: Explore Your Variables

Using the correlation matrix and availability plot above, choose a target variable and one or more predictors. Think about:

- Which variables are well-sampled enough to serve as predictors?
- Which target variable is scientifically interesting but sparse?
- Do the predictor-target correlations suggest a relationship worth modeling?

Plot the time-series of your chosen variables and scatter plots of each predictor against the target.
How many complete rows do you have when all your chosen variables are present?

In [ ]:
# ── Choose your variables ───────────────────────────────────────────
predictors =   # modify: choose your predictor variables from the candidates list above
target     =   # modify: choose your target variable from the candidates list above
selected   = predictors + [target]

# ── Temporal coverage ───────────────────────────────────────────────
fig, axes = plt.subplots(len(selected), 1, figsize=(12, 6), sharex=True)
for ax, var in zip(axes, selected):
    mask = df[var].notna()
    ax.scatter(df.loc[mask, 'time'], df.loc[mask, var], s=1, alpha=0.5)
    ax.set_ylabel(var)
    ax.text(0.98, 0.85, f'n={mask.sum():,}', transform=ax.transAxes, ha='right', fontsize=9)
axes[-1].set_xlabel('Time')
axes[0].set_title('Temporal coverage (0-30 m)')
plt.tight_layout()
plt.show()

# ── Predictor-target overlap ────────────────────────────────────────
mask = df[selected].notna().all(axis=1)
print(f"Complete rows with all variables present: {mask.sum():,}")

# ── Predictor-target relationships ──────────────────────────────────
fig, axes = plt.subplots(1, len(predictors), figsize=(5 * len(predictors), 4))
if len(predictors) == 1:
    axes = [axes]
sub = df.loc[mask]
for ax, var in zip(axes, predictors):
    ax.scatter(sub[var], sub[target], s=3, alpha=0.3)
    ax.set_xlabel(var)
    ax.set_ylabel(target)
plt.suptitle('Predictor-target relationships (0-30 m)')
plt.tight_layout()
plt.show()

## Preparing Training Data

### Train/Test Split

We keep only rows where all selected variables are present, then randomly shuffle and split 80/20 into training and test sets.
Because we are predicting from co-located measurements (not forecasting into the future), random splitting is appropriate here.
The model learns from the training set and is evaluated on held-out data it has never seen.

In [ ]:
mask = df[predictors + [target]].notna().all(axis=1)
data = df.loc[mask, predictors + [target]].values.copy()
print(f"Complete rows: {len(data):,}")

rng = np.random.default_rng(42)
data = data[rng.permutation(len(data))]

n_train = int(0.70 * len(data))  # modify: adjust train/val/test split ratios
n_val   = int(0.15 * len(data))

X_train = data[:n_train, :len(predictors)]
y_train = data[:n_train, len(predictors):]
X_val   = data[n_train:n_train+n_val, :len(predictors)]
y_val   = data[n_train:n_train+n_val, len(predictors):]
X_test  = data[n_train+n_val:, :len(predictors)]
y_test  = data[n_train+n_val:, len(predictors):]

print(f"Train: {X_train.shape[0]:,}, Val: {X_val.shape[0]:,}, Test: {X_test.shape[0]:,}")

#### Normalization

Neural networks train more effectively when inputs and targets have similar scales.
We subtract the mean and divide by the standard deviation, computed from the training set only.
The test set is normalized using the same training statistics to avoid data leakage.

In [ ]:
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0)
y_mean, y_std = y_train.mean(), y_train.std()

X_train_n = (X_train - X_mean) / X_std
X_val_n   = (X_val   - X_mean) / X_std
X_test_n  = (X_test  - X_mean) / X_std
y_train_n = (y_train - y_mean) / y_std
y_val_n   = (y_val   - y_mean) / y_std
y_test_n  = (y_test  - y_mean) / y_std

print(f"X_mean = {X_mean}")
print(f"X_std  = {X_std}")
print(f"y_mean = {y_mean:.2f}, y_std = {y_std:.2f}")

# ── Tensors ──────────────────────────────────────────────────────────
X_train_t = tf.constant(X_train_n, dtype=tf.float32)
X_val_t   = tf.constant(X_val_n,   dtype=tf.float32)
X_test_t  = tf.constant(X_test_n,  dtype=tf.float32)

y_train_t = tf.constant(y_train_n, dtype=tf.float32)
y_val_t   = tf.constant(y_val_n,   dtype=tf.float32)
y_test_t  = tf.constant(y_test_n,  dtype=tf.float32)

### Linear Regression Baseline
Before building a neural network, we fit a simple linear regression to establish a baseline.
If a linear model already performs well, the added complexity of an ANN may not be justified.

In [ ]:
### Linear Regression Baseline
model_lr = sm.OLS(y_train_n, sm.add_constant(X_train_n)).fit()
print(model_lr.summary())

# Test set performance
# Predictions (normalized → original scale)
y_pred_lr_test = model_lr.predict(sm.add_constant(X_test_n))
y_pred_lr = y_pred_lr_test * y_std + y_mean

# Metrics (original scale)
lr_r2_test = r2_score(y_test, y_pred_lr)
lr_rmse_test = np.sqrt(np.mean((y_test.flatten() - y_pred_lr.flatten())**2))

print(f"\nTest R² (LR):  {lr_r2_test:.4f}")
print(f"Test RMSE (LR): {lr_rmse_test:.4f}")

Examine the linear regression output. How much variance does it explain? Which predictors are significant? Is the gap between train and test performance large (overfitting) or small (underfitting)?

Can a neural network capture additional nonlinear structure?

### Building the Network

We build a simple feedforward network.
The architecture, learning rate, and training duration can all be adjusted.
Start simple and add capacity only if the model is underfitting.

#### Hyperparameters

A wider layer (more neurons) lets the network represent more complex relationships.
Deeper networks (more layers) can learn hierarchical patterns.
Start simple and add capacity only if the training loss suggests the model is underfitting.
If the gap between training and test loss grows, the network is too large for the available data.

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────
learning_rate = 1e-3 # modify: Adam learning rate
n_epochs      = 500 # modify: maximum training epochs
patience      = 30 # modify: stop if val_loss doesn't improve for this many epochs
batch_size    = 32 # modify: number of samples per gradient update

#### Build and Compile

We construct a feedforward network using `tf.keras.Sequential`.
Each hidden layer uses an activation function.
The output layer is linear (no activation) because we are predicting a continuous value.
The network is compiled with the Adam optimizer and mean squared error loss.

#### Train and Evaluate

Use `model.fit()` to train the network.
Pass in the training data, validation data, and an `EarlyStopping` callback that monitors `val_loss`.

In [ ]:
# ── Train the model ──────────────────────────────────────────────────


# ── Evaluate (train / val / test) in ORIGINAL SCALE ───────────────────
pred_train = denorm(model.predict(X_train_n, verbose=0), y_mean, y_std)
pred_val   = denorm(model.predict(X_val_n,   verbose=0), y_mean, y_std)
pred_test  = denorm(model.predict(X_test_n,  verbose=0), y_mean, y_std)

r2_train = r2_score(y_train, pred_train)
r2_val   = r2_score(y_val,   pred_val)
r2_test  = r2_score(y_test,  pred_test)

# RMSE (original units)
rmse_test = np.sqrt(np.mean((y_test - pred_test)**2))

# ── Report ───────────────────────────────────────────────────────────
print(f"Stopped at epoch: {len(hist.history['loss'])}")
print(f"Best val loss: {min(hist.history['val_loss']):.6f}\n")

print(f"Train R²: {r2_train:.4f}")
print(f"Val   R²: {r2_val:.4f}")
print(f"Test  R²: {r2_test:.4f}")
print(f"Test RMSE: {rmse_test:.4f}")

#### Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist.history['loss'], label='train')
ax.plot(hist.history['val_loss'], label='val')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('Training curve')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred_test, s=5, alpha=0.5)
lo = min(y_test.min(), pred_test.min())
hi = max(y_test.max(), pred_test.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8)
ax.set_xlabel(f'Observed {target}')
ax.set_ylabel(f'Predicted {target}')
ax.set_title(f'{target} prediction (R² = {r2_test:.2f})')
plt.tight_layout()
plt.show()

### Gap-Filling

With the trained network, we can estimate the target variable for every row where the predictors exist but the target was not measured.

In [ ]:
has_predictors = df[predictors].notna().all(axis=1)
missing_target = df[target].isna()
to_fill = has_predictors & missing_target

X_fill   = df.loc[to_fill, predictors].values
X_fill_n = (X_fill - X_mean) / X_std
pred_fill = denorm(model.predict(X_fill_n, verbose=0), y_mean, y_std)

print(f"Rows with predictors but missing {target}: {to_fill.sum():,}")
print(f"Predicted {target} range: {pred_fill.min():.1f} to {pred_fill.max():.1f}")
print(f"Training {target} range:  {y_train.min():.1f} to {y_train.max():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

obs_mask = df[target].notna()
ax.scatter(df.loc[obs_mask, 'time'], df.loc[obs_mask, target], s=3, alpha=0.4,
           color='steelblue', label=f'observed (n={obs_mask.sum():,})')

ax.scatter(df.loc[to_fill, 'time'], pred_fill, s=1, alpha=0.2,
           color='tomato', label=f'predicted (n={to_fill.sum():,})')

ax.set_xlabel('Time')
ax.set_ylabel(target)
ax.set_title(f'{target}: observed vs network-predicted')
ax.legend()
plt.tight_layout()
plt.show()

Does the predicted range cover the full range of observed values? Are there time periods where the predictions look suspect? What additional predictors might improve the result?

### Explore Other Combinations

The same approach can be applied to other target variables.
Use the helper below to check how much training data is available for different predictor/target combinations before rebuilding the model above.

In [ ]:
def check_overlap(df, predictors, target):
    cols = predictors + [target]
    n = df[cols].notna().all(axis=1).sum()
    print(f"Predictors: {predictors}")
    print(f"Target:     {target}")
    print(f"Complete rows: {n:,}\n")

# modify: try different predictor/target combinations
check_overlap(df, predictors=['temp', 'csal'], target='alk')